In [2]:
include("Quadprog/src/Quadprog.jl")
using .Quadprog
using LinearAlgebra

In [12]:
# Day Ahead Market

wind = 0
# Second order cost coefficients 
P   = 2 * [0.11 0 0; 0 0.085 0; 0 0 0.1225]
# First order cost coefficients 
q   = [5 1.2 1]'
# Generation bounds
lb  = [0 0 0]'
ub  = [250 300 270]'
# Generation equality
Aeq = [1 1 1]      
beq = 600 - wind
# Solve system using quadprog
(x, fx, status) = quadprog(P, q; lb = lb, ub = ub, Aeq = Aeq, beq = beq)

println("x = ", x)
println("fx = ", fx)
println("status = ", status)
println("sum(x) = ", sum(x))

x = [175.84806318164723, 249.92102294095525, 174.23091387739754]
fx = 13782.660398646109
status = 0
sum(x) = 600.0


In [4]:
mc = P * x .+ vec(q)
println(mc) 

[42.308160962767964, 42.308160962767964, 42.30816096276796]


In [13]:
# Real time market

N = 3
Pg = [175.84806318164723 249.92102294095525 174.23091387739754]'
# Second order cost coefficients
a      = [0.22, 0.17, 0.245]    
# First order cost coefficients      
B      = [5.0, 1.2, 1.0]  
# Up/down regulation factors            
kup    = 1.2                          
kdown  = 0.8
# Day ahead/real time generation
D_da   = 600.0
D_rt   = 560.0                        

Da = Diagonal(a)
Bup, Bdown = kup .* B, kdown .* B

# Change in generation
ΔD = D_rt - D_da

Qh = [Matrix(Da) -Matrix(Da); -Matrix(Da) Matrix(Da)]     
c  = vcat(a .* Pg .+ Bup, -a .* Pg .- Bdown)

lb = [0 0 0 0 0 0]'                                         
ub  = [250 300 270 250 300 270]'

# Solve system using quadprog
(x, fx, st) = quadprog(Qh, c; Aeq = [ones(1, N) -ones(1, N)],beq = ΔD, lb = lb, ub=ub)
rup, rdown = x[1:N], x[N+1:end]
g = Pg .+ rup .- rdown

mc_up   = a .* g .+ Bup       
mc_down = a .* g .+ Bdown   

println("New dispatch g        = ", g)
println("MC up   [\$/MWh]       = ", mc_up)
println("MC down [\$/MWh]       = ", mc_down)


New dispatch g        = [165.74050394885293; 232.37006393380972; 161.88943211733735;;]
MC up   [$/MWh]       = [42.46291086874765; 40.94291086874765; 40.862910868747655;;]
MC down [$/MWh]       = [40.46291086874765; 40.462910868747656; 40.46291086874765;;]
